# Module 4 | Class 4 Assignment
## SVM vs KNN Showdown — Telco Customer Churn

**Maqsad:** SVM va KNN klassifikatorlarini Telco Churn datasetida solishtirish, giperparametrlarni o'rganish va qaysi algoritmni qachon ishlatishni tushunish.

---

## Task 1: Ma'lumotlarni Tayyorlash va Masshtablash

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

# ── Dataset yuklash ────────────────────────────────────────────────────────
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Dataset shakli:", df.shape)

# ── Tozalash ───────────────────────────────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── One-Hot Encoding ───────────────────────────────────────────────────────
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), columns=cat_cols, drop_first=True)
df_encoded = df_encoded.fillna(0)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']
print(f"Features: {X.shape[1]}  |  Samples: {X.shape[0]}")

In [ ]:
# ── Train/Test bo'lish ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# ── Masshtablash — SVM va KNN uchun MAJBURIY ──────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"\n✅ Masshtablash tugadi.")
print(f"   O'rtacha: {X_train_s.mean():.6f}  (≈ 0)")
print(f"   Std:      {X_train_s.std():.6f}   (≈ 1)")

## Task 2: SVM Klassifikatori (RBF Kernel)

In [ ]:
# ── SVM — RBF kernel ──────────────────────────────────────────────────────
start = time.time()
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_s, y_train)
svm_time = time.time() - start

y_pred_svm = svm.predict(X_test_s)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1  = f1_score(y_test, y_pred_svm)

print("=" * 48)
print("📊 SVM (RBF Kernel) Natijalari")
print("=" * 48)
print(f"  Accuracy      : {svm_acc:.4f}")
print(f"  F1-Score      : {svm_f1:.4f}")
print(f"  O'qitish vaqti: {svm_time:.2f}s")
print()
print(classification_report(y_test, y_pred_svm, target_names=['No Churn', 'Churn']))

## Task 3: KNN Klassifikatori (K=5)

In [ ]:
# ── KNN — K=5 ─────────────────────────────────────────────────────────────
start = time.time()
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_s, y_train)
knn5_time = time.time() - start

y_pred_knn5 = knn5.predict(X_test_s)
knn5_acc = accuracy_score(y_test, y_pred_knn5)
knn5_f1  = f1_score(y_test, y_pred_knn5)

print("=" * 48)
print("📊 KNN (K=5) Natijalari")
print("=" * 48)
print(f"  Accuracy      : {knn5_acc:.4f}")
print(f"  F1-Score      : {knn5_f1:.4f}")
print(f"  O'qitish vaqti: {knn5_time:.4f}s  (≈ 0 — faqat data saqlaydi)")
print()
print(classification_report(y_test, y_pred_knn5, target_names=['No Churn', 'Churn']))

## Task 4: Turli K Qiymatlari Tahlili (K = 3, 5, 10)

In [ ]:
# ── K=3, 5, 10 ────────────────────────────────────────────────────────────
k_results = []
for k in [3, 5, 10]:
    start = time.time()
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    t = time.time() - start
    yp = knn.predict(X_test_s)
    acc = accuracy_score(y_test, yp)
    f1  = f1_score(y_test, yp)
    k_results.append({
        'K qiymati': k,
        'Accuracy': round(acc, 4),
        'F1-Score': round(f1, 4),
        "Vaqt (s)": round(t, 4)
    })

k_df = pd.DataFrame(k_results)
print(k_df.to_string(index=False))

In [ ]:
# ── K qiymatlari vizualizatsiya ───────────────────────────────────────────
k_vals   = [r['K qiymati'] for r in k_results]
acc_vals = [r['Accuracy']  for r in k_results]
f1_vals  = [r['F1-Score']  for r in k_results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_vals, acc_vals, 'o-', color='#3498db', lw=2, ms=9, label='Accuracy')
ax.plot(k_vals, f1_vals,  's-', color='#e67e22', lw=2, ms=9, label='F1-Score')
for k, a, f in zip(k_vals, acc_vals, f1_vals):
    ax.annotate(f'{a:.4f}', (k, a), textcoords='offset points', xytext=(6,  7), color='#3498db', fontsize=10)
    ax.annotate(f'{f:.4f}', (k, f), textcoords='offset points', xytext=(6, -14), color='#e67e22', fontsize=10)
ax.set_xlabel("K qiymati", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("KNN — K qiymati ta'siri (Accuracy vs F1)", fontsize=13, fontweight='bold')
ax.set_xticks(k_vals)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**K qiymatlari tahlili:**

| K | Accuracy | F1-Score |
|---|----------|----------|
| 3 | 0.6246 | 0.3754 |
| 5 | 0.6388 | 0.3677 |
| 10 | 0.6707 | 0.2948 |

- **K=3** — kichik K → shovqinga sezgir (overfitting xavfi), lekin bu datasetda F1 bo'yicha eng yaxshi
- **K=5** — muvozanatlangan, lekin K=3 dan bir oz past
- **K=10** — katta K → silliqroq chegara (underfitting xavfi), F1 eng past
- **Eng yaxshi K = 3** (F1 = 0.3754)

## Task 5: SVM va Eng Yaxshi KNN — To'liq Solishtirma

In [ ]:
# ── Yakuniy solishtirma jadvali ───────────────────────────────────────────
summary = pd.DataFrame({
    'Model':           ['SVM (RBF kernel)', 'KNN (K=3)'],
    'Accuracy':        [0.6849, 0.6246],
    'F1-Score':        [0.3639,  0.3754],
    "O'qitish vaqti": [f'1.53s', '≈ 0.00s'],
})
print(summary.to_string(index=False))

In [ ]:
# ── Vizual solishtirma ────────────────────────────────────────────────────
models = ['SVM (RBF)', 'KNN (K=3)']
accs   = [0.6849, 0.6246]
f1s    = [0.3639,  0.3754]

x = np.arange(len(models))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#3498db', alpha=0.87)
b2 = ax.bar(x + w/2, f1s,  w, label='F1-Score',  color='#e74c3c', alpha=0.87)
ax.bar_label(b1, fmt='%.4f', padding=4, fontsize=10)
ax.bar_label(b2, fmt='%.4f', padding=4, fontsize=10)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('SVM vs KNN — Accuracy va F1 Solishtirmasi', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(models, fontsize=12)
ax.set_ylim(0, 0.9)
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Eslatma:** KNN "o'qitishi" aslida ma'lumotlarni xotirada saqlashdan iborat — shuning uchun vaqti deyarli nol. Ammo bashorat vaqtida har bir test namunasi uchun barcha o'quv namunalari bilan masofa hisoblanadi, bu esa katta datasetlarda sekinlikka olib keladi.

## Task 6: Muhokama — Qachon KNN, Qachon SVM?

### Natijalar Xulosasi

| Model | Accuracy | F1-Score | O'qitish vaqti |
|-------|----------|----------|----------------|
| SVM (RBF) | 0.6849 | 0.3639 | 1.53s |
| KNN (K=3) | 0.6246 | 0.3754 | ≈ 0.00s |
| KNN (K=5) | 0.6388 | 0.3677 | ≈ 0.00s |
| KNN (K=10) | 0.6707 | 0.2948 | ≈ 0.00s |

---

### 🧠 KNN ni SVM dan afzal ko'rgan hollar

**1. Kichik va o'rta hajmli datasetlar:**
KNN o'qitish bosqichida faqat ma'lumotlarni xotirada saqlaydi — o'qitish deyarli zudlik bilan tugaydi. Ammo bashorat vaqtida har bir test namunasi uchun barcha o'quv namunalari bilan masofa hisoblanadi. Agar dataset kichik bo'lsa (masalan, 10 000 dan kam), bu muammo emas va KNN amaliy ishlaydi.

**2. Tushuntirish qulayligi (Interpretability):**
KNN ning ish mantiqini oddiy so'z bilan tushuntirish oson: *"Bu mijoz ketadimi? Unga eng o'xshash 3 ta mijozga qaraymiz — ularning ko'pchiligi ketgan bo'lsa, bu ham ketadi."* Biznes jamoalar yoki mijozlarga modelni tushuntirish kerak bo'lganda KNN ancha qulay.

**3. Murakkab chegara shakllari:**
SVM aniq kernel funksiyasini tanlashni talab qiladi. KNN esa hech qanday model strukturasini talab qilmaydi — murakkab va notekis chegara shakllarini ham organik ravishda ushlab oladi.

**4. Tez prototiplash:**
Giperparametr sifatida faqat K ni sozlash kifoya. SVM da esa kernel, C, gamma kabi bir necha parametrni o'rganish kerak — bu ko'proq vaqt oladi.

---

### ⚠️ SVM ni afzal ko'rgan hollar

- **Yuqori o'lchamli ma'lumotlar** (ko'p feature): SVM yuqori o'lchamda ham yaxshi ishlaydi; KNN esa "o'lchamlar la'nati" (*curse of dimensionality*) dan aziyat chekadi — masofalar ma'nosizlashib qoladi.
- **Katta dataset, tez bashorat kerak**: SVM o'qitishi sekin, lekin bashorat tez. KNN bashorati esa o'lcham bilan chiziqli ravishda sekinlashadi.
- **Xotirani tejash**: KNN barcha o'quv ma'lumotlarini saqlaydi; SVM faqat "support vector"larni xotirada ushlab turadi.

---

### ✅ Xulosa

Bu topshiriqda SVM Accuracy bo'yicha (0.6849) KNN dan biroz ustun keldi, F1 bo'yicha esa natijalar juda yaqin. Real loyihada tanlov dataset hajmi, tushuntirish talablari va ishlash tezligiga bog'liq. Dataset kichik va mijozga tushuntirish kerak bo'lsa — **KNN** afzal. Katta, yuqori o'lchamli ma'lumotlarda esa **SVM** yoki gradient boosting kabi kuchliroq modellar to'g'riroq tanlov bo'ladi.